# 🧩 NumPy: Additional Topics — Gap-Fill Companion

This notebook covers the remaining NumPy topics not in the main `NumPy_Zero_to_Master.ipynb`:
empty arrays, file I/O, random shuffling, partial sort, splitting, comparisons, searching,
bincount, and string operations (`numpy.char`). Same format: theory → example → your turn → solution.

**Do this after** `NumPy_Zero_to_Master.ipynb`.


In [ ]:
import numpy as np
print(np.__version__)

---
## 1 — Create an Empty Array

📖 `np.empty(shape)` allocates memory **without initializing values** — faster than `zeros`/
`ones`, but contents are garbage until you fill them. Use it only when you'll overwrite every
element anyway.

In [ ]:
e = np.empty((2, 3))
print("empty (garbage values):\n", e)
e[:] = 0   # you must fill it yourself if you need known values
print("after filling:\n", e)

⚠️ **Trap.** Never read from `np.empty` before writing to it — the values are whatever was
already in that memory. If you need zeros, just use `np.zeros`.

### ✏️ Your Turn — create a (3,3) empty array, then fill it with 7 using `.fill(7)`.

In [ ]:
arr = None
print(arr)

✅ **Solution**
```python
arr = np.empty((3,3)); arr.fill(7)
```

---
## 2 — Reading and Writing Files (binary + text)

📖 Two families:
- **Binary, NumPy-native:** `np.save("f.npy", arr)` / `np.load("f.npy")` — fast, exact dtype preserved.
- **Text/CSV:** `np.savetxt("f.csv", arr, delimiter=",")` / `np.loadtxt("f.csv", delimiter=",")` —
  human-readable, portable, but slower and loses exact dtype (usually loads as float).

In [ ]:
arr = np.array([[1,2,3],[4,5,6]])

np.save("/tmp/arr.npy", arr)
loaded_npy = np.load("/tmp/arr.npy")
print("npy roundtrip matches:", np.array_equal(arr, loaded_npy))

np.savetxt("/tmp/arr.csv", arr, delimiter=",", fmt="%d")
loaded_csv = np.loadtxt("/tmp/arr.csv", delimiter=",")
print("csv roundtrip:\n", loaded_csv, loaded_csv.dtype)

⚡ **Pro tip.** Use `.npy`/`.npz` for intermediate/checkpoint data within Python (fast, exact);
use `.csv` only when a human or another tool needs to read it.

### ✏️ Your Turn — save a (4,) float array to `/tmp/vec.npy`, reload it, confirm equality.

In [ ]:
vec = np.array([1.5, 2.5, 3.5, 4.5])
# save, reload, compare


✅ **Solution**
```python
np.save("/tmp/vec.npy", vec)
reloaded = np.load("/tmp/vec.npy")
print(np.array_equal(vec, reloaded))
```

---
## 3 — Random Shuffle

📖 `np.random.shuffle(arr)` shuffles **in-place** along the first axis. `np.random.permutation(arr)`
returns a **new shuffled copy** (original untouched) — and if given an integer `n`, returns a
shuffled `arange(n)`, handy for shuffling indices.

In [ ]:
np.random.seed(0)
a = np.arange(5)
np.random.shuffle(a)          # in-place
print("shuffled in place:", a)

b = np.arange(5)
c = np.random.permutation(b)  # new array, b untouched
print("original untouched:", b, " permuted copy:", c)

# common pattern: shuffle indices to shuffle multiple related arrays together
idx = np.random.permutation(5)
X = np.arange(10).reshape(5,2)
y = np.array([0,1,0,1,1])
print("X shuffled:\n", X[idx], "\ny shuffled:", y[idx])

⚠️ **Trap.** `shuffle` mutates its argument and returns `None` — `x = np.random.shuffle(x)` will
set `x` to `None`! Use `permutation` when you want a return value.

### ✏️ Your Turn — shuffle the indices of a 6-row array and apply the same order to two arrays.

In [ ]:
A = np.arange(6); B = np.arange(6)*10
# shuffle both with the same random order


✅ **Solution**
```python
idx = np.random.permutation(6)
A_shuf, B_shuf = A[idx], B[idx]
```

---
## 4 — Transpose of a 1D Array (a special case)

📖 `.T` on a **1D** array is a **no-op** — there's only one axis, nothing to swap. To get a
"column vector" you must add a dimension first (`reshape(-1,1)` or `[:, None]`).

In [ ]:
v = np.array([1,2,3])
print("1D transpose == original:", np.array_equal(v.T, v))   # True -- no effect!

col = v.reshape(-1, 1)   # now truly 2D, a column vector
print("column vector shape:", col.shape)
print(col)

### ✏️ Your Turn — confirm `v[:, None]` gives the same column-vector shape as `v.reshape(-1,1)`.

In [ ]:
v = np.array([4,5,6])
same_shape = None
print(same_shape)

✅ **Solution**
```python
same_shape = v.reshape(-1,1).shape == v[:, None].shape   # True, both (3,1)
```

---
## 5 — Partial Sort

📖 `np.partition(arr, k)` rearranges so the **k-th smallest** element is in its sorted position,
with everything smaller before it and everything larger after (each side unsorted). Much faster
than a full sort when you only need "top-k" or "k-th smallest", since it's O(n) not O(n log n).

In [ ]:
a = np.array([9, 1, 8, 2, 7, 3, 6, 4, 5])
part = np.partition(a, 3)
print("partitioned around index 3:", part)
print("the 4 smallest values (unordered):", part[:4])

# common use: get the 3 largest values fast
top3 = np.partition(a, -3)[-3:]
print("3 largest (unordered):", top3)

### ✏️ Your Turn — get the 4 smallest values of `data` using partition (don't fully sort).

In [ ]:
data = np.array([50, 12, 88, 3, 45, 9, 71, 2])
four_smallest = None
print(four_smallest)

✅ **Solution**
```python
four_smallest = np.partition(data, 4)[:4]
```

---
## 6 — Split

📖 `np.split(arr, n)` divides an array into `n` **equal** parts (raises an error if it can't
divide evenly). `np.array_split(arr, n)` allows uneven splits. You can also split at specific
index positions.

In [ ]:
a = np.arange(9)
parts = np.split(a, 3)          # 3 equal parts of 3
print("equal split:", parts)

b = np.arange(10)
uneven = np.array_split(b, 3)   # 10 doesn't divide evenly by 3
print("uneven split:", uneven)

# split at specific positions
at_positions = np.split(a, [2, 5])   # cut before index 2 and before index 5
print("split at [2,5]:", at_positions)

⚠️ **Trap.** `np.split(arr, 3)` on a length-10 array raises `ValueError` because 10 isn't
divisible by 3 — use `array_split` for uneven lengths.

### ✏️ Your Turn — split a length-12 array into 4 equal parts, and separately split a length-7
array into 3 parts using `array_split`.

In [ ]:
a12 = np.arange(12); a7 = np.arange(7)
parts4 = None
parts3 = None
print(parts4); print(parts3)

✅ **Solution**
```python
parts4 = np.split(a12, 4)
parts3 = np.array_split(a7, 3)
```

---
## 7 — Compare with a Scalar & Elementwise Comparison

📖 Comparison operators (`>`, `<`, `==`, `!=`) broadcast against arrays and return a **boolean
array**, elementwise. Comparing two arrays of the same shape compares position-by-position.

In [ ]:
a = np.array([1, 5, 3, 8, 2])

# compare with a scalar
print("a > 3:", a > 3)
print("a == 5:", a == 5)

# elementwise compare two arrays
b = np.array([2, 4, 3, 9, 1])
print("a > b elementwise:", a > b)
print("a == b elementwise:", a == b)

# np.array_equal checks whole-array equality (returns one bool)
print("arrays fully equal?", np.array_equal(a, b))

### ✏️ Your Turn — given two arrays, find the positions (indices) where they differ.

In [ ]:
x = np.array([1,2,3,4,5]); y = np.array([1,0,3,0,5])
diff_positions = None
print(diff_positions)

✅ **Solution**
```python
diff_positions = np.where(x != y)[0]   # array([1, 3])
```

---
## 8 — Search: `searchsorted`

📖 `np.searchsorted(sorted_arr, values)` finds the index where each value **would be inserted**
to keep the array sorted — a fast binary search (O(log n)), useful for lookups/binning.

In [ ]:
sorted_arr = np.array([1, 3, 5, 7, 9])
print("insert 4 at index:", np.searchsorted(sorted_arr, 4))   # between 3 and 5 -> index 2
print("insert [0, 6, 10]:", np.searchsorted(sorted_arr, [0, 6, 10]))

# practical use: bucket/bin a value into predefined ranges
bins = np.array([0, 60, 70, 80, 90, 100])   # grade boundaries
grades = np.array(["F","D","C","B","A"])
score = 85
bucket = np.searchsorted(bins, score, side="right") - 1
print(f"score {score} -> grade {grades[bucket]}")

### ✏️ Your Turn — use `searchsorted` to find where 15 and 100 would insert into
`[10, 20, 30, 40, 50]`.

In [ ]:
boundaries = np.array([10,20,30,40,50])
positions = None
print(positions)

✅ **Solution**
```python
positions = np.searchsorted(boundaries, [15, 100])   # [1, 5]
```

---
## 9 — `bincount`

📖 `np.bincount(arr)` counts occurrences of each **non-negative integer** — index `i` of the
result is the count of `i` in the input. Great for fast histograms of small-integer data (like
category codes, dice rolls, or class labels).

In [ ]:
rolls = np.array([1,3,3,6,2,1,1,5,3,6,6,6])
counts = np.bincount(rolls)   # index 0 unused here since no zeros
print("counts per face value:", counts)
for face in range(1,7):
    print(f"  face {face}: {counts[face]} times")

# weighted bincount: sum weights per bin instead of counting
weights = np.array([10,20,30,5,15,25,40,1,2,3,4,5])
weighted = np.bincount(rolls, weights=weights)
print("\nweighted sums per face:", weighted)

### ✏️ Your Turn — count how many times each class label (0,1,2) appears in `labels`.

In [ ]:
labels = np.array([0,1,1,2,0,2,2,2,1,0,0])
class_counts = None
print(class_counts)

✅ **Solution**
```python
class_counts = np.bincount(labels)   # [4, 3, 4]
```

---
## 10 — `numpy.char` Module: String Operations & Translate

📖 `numpy.char` applies string methods **elementwise** across a whole array of strings — instead
of looping in Python. Mirrors Python's `str` methods: `.upper`, `.lower`, `.strip`, `.replace`,
plus `np.char.translate` for character-mapping (like Python's `str.translate`).

In [ ]:
words = np.array(["  Hello  ", "WORLD", "NumPy"])

print("strip:", np.char.strip(words))
print("lower:", np.char.lower(words))
print("upper:", np.char.upper(words))
print("replace 'l' with 'L':", np.char.replace(words, "l", "L"))
print("string length of each:", np.char.str_len(words))

# translate: map specific characters (like a substitution cipher)
table = str.maketrans("aeiou", "@3!0*")
vowel_swap = np.array([s.translate(table) for s in ["banana", "orange", "grape"]])
print("\ntranslated (vowel substitution):", vowel_swap)

⚡ **Pro tip.** `numpy.char` functions are vectorized versions of Python string methods —
useful when you have a large NumPy array of strings and want to avoid a slow Python `for` loop.
For heavy text work with tabular data, Pandas' `.str` accessor (covered in the Pandas lab) is
usually more convenient.

### ✏️ Your Turn — given an array of names, strip whitespace and title-case each one.

In [ ]:
names = np.array(["  alice", "BOB  ", " carla "])
cleaned = None
print(cleaned)

✅ **Solution**
```python
cleaned = np.char.title(np.char.strip(names))
```

---
🎉 **Gap-fill complete.** Combined with `NumPy_Zero_to_Master.ipynb`, you now have full coverage:
array creation (incl. empty), file I/O (npy + csv), random sampling & shuffling, indexing
(basic/integer/boolean), transpose (incl. 1D edge case), full & partial sort, concatenate/stack/
split, scalar & elementwise comparison, search (where + searchsorted), unique + bincount, math &
linear algebra, and `numpy.char` string operations.

### 📌 Quick-Reference (this notebook)
`np.empty, .fill` · `np.save/np.load, np.savetxt/np.loadtxt` · `np.random.shuffle,
np.random.permutation` · 1D `.T` is a no-op, use `reshape(-1,1)` · `np.partition` ·
`np.split, np.array_split` · `>,<,==,!=, np.array_equal` · `np.searchsorted` · `np.bincount` ·
`np.char.upper/lower/strip/replace/str_len, np.char.title, str.translate`
